In [1]:
API_AQ = "914e962d8151116529b100714b2e5786d9cda8cedaee960bec6eff79cf511d15"
API_OW = "384724f767e320a1f74337f1e1c0e719"

## Timeline for the dataset 
- training period: 2 years, from may 2023 to April 2025
- may 2025: validation data for v1 model (tuning)
- june: test data for v1 model (unseen)
- july - december 2025: simulated incoming of new data for model drift monitoring




In [5]:
cities = {"Paris": "48.8566,2.3522"}
start_train_date_str = "2023-05-01"
start_project_date_str = "2023-05-01"
end_train_date_str = "2025-04-30"
end_project_date_str = "2025-12-31"

In [2]:
import requests
import pandas as pd
import numpy as np
import json
import time
import os
from tqdm.auto import tqdm
from pathlib import Path
from src.params import *


## Download OpenAQ data 

### Find locations
In this API, you need to 

i. identify location to get the sensor ids 

ii. retrieve measurements by sensors. 


In [3]:
# first, get locations for Paris
url_loc = "https://api.openaq.org/v3/locations"

params_loc = {"parameters_id" : 2, #PM2.5 sensors
               "coordinates": "40.8169,-73.9558", # Paris Lat Lon
               "radius": 5000,# 5km radius
               "limit": 1000
              }

header_loc = {"X-API-key": API_AQ}
data_loc = requests.get(url= url_loc, params= params_loc, headers= header_loc).json()


### Get sensors ID 

In [13]:

#filter for monitor grade sensors
monitor_loc = [loc for loc in data_loc["results"] if loc["isMonitor"]]

#filter for sensor still alive
end_project_date_dt = pd.to_datetime(end_project_date_str, utc= True)
alived_loc = [loc for loc in monitor_loc if pd.to_datetime(loc["datetimeLast"]["utc"]) >= end_project_date_dt]

#filter for availability over training set
start_project_date_dt = pd.to_datetime(start_project_date_str, utc= True)
full_time_loc = [loc for loc in alived_loc if pd.to_datetime(loc["datetimeFirst"]["utc"]) <= start_project_date_dt]


print("============")
print(f"\n Initial number of points of measurement: {len(monitor_loc)}")
print(f"Number of points of measurement still alive at end of dataset: {len(alived_loc)}")
print(f"Number of points of measurement available throughout dataset: {len(full_time_loc)}")



 Initial number of points of measurement: 5
Number of points of measurement still alive at end of dataset: 5
Number of points of measurement available throughout dataset: 5


In [6]:
#get pm2.5 sensors id
sensors_id = []

for loc in full_time_loc:
    for sensor in loc["sensors"]:
        if sensor["parameter"]["id"] == 2:
            sensors_id.append(sensor["id"])

In [9]:
print(f"number of available sensors: {len(sensors_id)}")

number of available sensors: 3


### Get sensor data

In [7]:

sensor_data = {}
for sensor_id in sensors_id:
    url_sensors = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"
    params_sensors = {
                    "datetime_from": start_train_date_str,
                    "datetime_to": end_train_date_str,
                    "limit": 1000,


    }

    results_sensor = requests.get(url_sensors, params= params_sensors, headers = header_loc).json()
    sensor_data[sensor_id] = results_sensor


In [10]:
params_sensors = {
                    "datetime_from": "2023-06-08",
                    "datetime_to": "2023-07-05",
                    "limit": 1000,


    }

sensor_id = 14798912
url_rio2 = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"
url_rio = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"
results_rio = requests.get(url_rio2, params= params_sensors, headers = header_loc).json()
results_rio


{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 1000,
  'found': 73},
 'results': [{'value': 5.26,
   'flagInfo': {'hasFlags': False},
   'parameter': {'id': 2,
    'name': 'pm25',
    'units': 'µg/m³',
    'displayName': None},
   'period': {'label': '1day',
    'interval': '24:00:00',
    'datetimeFrom': {'utc': '2025-12-04T03:00:00Z',
     'local': '2025-12-04T00:00:00-03:00'},
    'datetimeTo': {'utc': '2025-12-05T03:00:00Z',
     'local': '2025-12-05T00:00:00-03:00'}},
   'coordinates': None,
   'summary': {'min': 4.13,
    'q02': 4.193,
    'q25': 4.985,
    'median': 5.234999999999999,
    'q75': 5.369999999999999,
    'q98': 6.702399999999999,
    'max': 6.89,
    'avg': 5.2562500000000005,
    'sd': 0.8195273027788639},
   'coverage': {'expectedCount': 24,
    'expectedInterval': '24:00:00',
    'observedCount': 8,
    'observedInterval': '08:00:00',
    'percentComplete': 33.0,
    'percentCoverage': 33.0,
    'datetimeFrom': {'utc': '2025-12-04T20:0

 ### format data in df 

In [9]:
all_dataframes= []

for sensor_id, data in sensor_data.items():
    rows = [] #empty list to add required fields for a given sensor_id

    #iterates over measurements for that sensor
    for results in data["results"]:
        rows.append({
            "sensor_id": sensor_id,
            "date_from_utc": results["period"]['datetimeFrom']['utc'],
            "date_from_local": results["period"]['datetimeFrom']['local'],
            "date_to_utc": results["period"]['datetimeTo']['utc'],
            "date_to_local": results["period"]['datetimeTo']['local'],
            "pm25_avg": results["value"],
            "pm25_min": results["summary"]["min"],
            "pm25_q25": results["summary"]["q25"],
            "pm25_median": results["summary"]["median"],
            "pm25_q75": results["summary"]["q75"],
            "pm25_max": results["summary"]["max"],
            "coverage": results["coverage"]["percentComplete"]
        })
    df_sensor = pd.DataFrame(rows)
    all_dataframes.append(df_sensor)

df_aq_by_city = pd.concat(all_dataframes, ignore_index= True)
df_aq_by_city


,sensor_id,date_from_utc,date_from_local,date_to_utc,date_to_local,pm25_avg,pm25_min,pm25_q25,pm25_median,pm25_q75,pm25_max,coverage
0,1582598,2023-04-30T22:00:00Z,2023-05-01T00:00:00+02:00,2023-05-01T22:00:00Z,2023-05-02T00:00:00+02:00,16.00,4.9,10.675,15.05,19.650,32.0,100.0
1,1582598,2023-05-01T22:00:00Z,2023-05-02T00:00:00+02:00,2023-05-02T22:00:00Z,2023-05-03T00:00:00+02:00,12.60,6.6,9.650,10.90,15.600,21.9,100.0
2,1582598,2023-05-02T22:00:00Z,2023-05-03T00:00:00+02:00,2023-05-03T22:00:00Z,2023-05-04T00:00:00+02:00,11.60,4.8,9.900,12.25,13.800,17.6,100.0
3,1582598,2023-05-03T22:00:00Z,2023-05-04T00:00:00+02:00,2023-05-04T22:00:00Z,2023-05-05T00:00:00+02:00,8.96,3.9,7.825,9.40,10.700,11.9,100.0
4,1582598,2023-05-04T22:00:00Z,2023-05-05T00:00:00+02:00,2023-05-05T22:00:00Z,2023-05-06T00:00:00+02:00,5.07,0.4,3.875,4.65,6.425,9.6,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2043,1562368,2025-04-24T22:00:00Z,2025-04-25T00:00:00+02:00,2025-04-25T22:00:00Z,2025-04-26T00:00:00+02:00,15.50,12.8,14.125,15.45,16.775,18.1,8.0
2044,1562368,2025-04-25T22:00:00Z,2025-04-26T00:00:00+02:00,2025-04-26T22:00:00Z,2025-04-27T00:00:00+02:00,14.80,13.5,14.125,14.75,15.375,16.0,8.0
2045,1562368,2025-04-26T22:00:00Z,2025-04-27T00:00:00+02:00,2025-04-27T22:00:00Z,2025-04-28T00:00:00+02:00,14.30,9.9,12.600,15.30,16.500,17.7,13.0
2046,1562368,2025-04-27T22:00:00Z,2025-04-28T00:00:00+02:00,2025-04-28T22:00:00Z,2025-04-29T00:00:00+02:00,20.30,18.8,19.550,20.30,21.050,21.8,8.0


### Quality check of data for a given city

In [10]:
#nb of days where at least one sensor produced data
df_aq_by_city["date_from_local"].nunique()
print(f"missing {365*2 - df_aq_by_city["date_from_local"].nunique()} days")
print(f"the dataset is {df_aq_by_city["date_from_local"].nunique() / (365*2)*100}% complete")

missing 34 days
the dataset is 95.34246575342465% complete


In [11]:
df_aq_by_city["pm25_avg"].describe()
#no clear sign of numerical encoding for missing values (-1, 999)
#float64 so no categorical encoding for missing valules (? , *)


count    2048.000000
mean        9.339415
std         6.498096
min        -1.140000
25%         5.150000
50%         8.290000
75%        12.500000
max        62.500000
Name: pm25_avg, dtype: float64

In [12]:
#nb of neg (aberrant?) values
aberrant_measurements = (df_aq_by_city["pm25_avg"] < 0).sum()
print(f"there are {aberrant_measurements / df_aq_by_city.shape[0]*100}% of aberrant measurements")
print(f"there are {df_aq_by_city["pm25_avg"].isna().sum()} Nan")

there are 4.736328125% of aberrant measurements
there are 0 Nan


In [13]:
#======================
# GAPS IN MEASUREMENTS
#======================
#no NaN, so all availables date means at least one sensor measured.

#convert to dt
df_aq_by_city["date"] = pd.to_datetime(df_aq_by_city["date_from_utc"], utc= True)

#extract unique dates
dates_unique = pd.Series(df_aq_by_city["date"].unique())
#sort
dates_unique = dates_unique.sort_values(ascending= True)

#calculate delta and extract gaps (more than 1 diff)
days_diff = dates_unique.diff().dt.days
days_gaps = days_diff[days_diff > 1]

print(f" Number of gaps: {(days_diff > 1).sum()}")
print(f"average duration of gasp {days_gaps.mean()}")
print(f"max duration of gaps: {days_gaps.max()}")

 Number of gaps: 17
average duration of gasp 3.0
max duration of gaps: 7.0


## Download weather data

In [14]:
# parameters for API call
url = "https://api.openweathermap.org/data/3.0/onecall/day_summary"
city = "Paris"
cache_dir = f"../data/cache/{city}"
os.makedirs(cache_dir, exist_ok= True)
max_retry = 3


In [15]:
start_train_date_str = "2023-05-01"
end_train_date_str = "2023-05-14"

In [ ]:
# =======================
# FETCH OPENWEATHER DATA
# =======================


date_range = pd.date_range(start=start_train_date_str, end=end_train_date_str, freq="D")
successful_call = 0  # Counter of successful new fetches
cached_count = 0     # Counter of already fetched and cached data

# --- Main loop: iterate through date range ---
for day in tqdm(date_range, desc="Fetching weather data"):
    day_str = day.strftime('%Y-%m-%d')
    cache_file = f"{cache_dir}/weather_{day_str}.json"

    # Check if data already cached
    if os.path.exists(cache_file):
        cached_count += 1 #if exists, it means that this day was successfully fetched before
        continue

    # --- Prepare API call parameters ---
    params = {
        "lat": 48.8566,
        "lon": 2.3522, #paris lat/lon
        "date": day_str,
        "appid": API_OW,
        "units": "metric"
    }

    # --- Retry loop: attempt API call up to max_retry times ---
    for attempt in range(max_retry):
        response = requests.get(url, params)

        # Handle rate limit error (HTTP 429)
        if response.status_code == 429:
            if attempt < max_retry - 1:
                tqdm.write(f"⚠️ Attempt {attempt}/{max_retry} failed: rate limit. Will retry in 60s")
                time.sleep(60)
                continue
            else:
                tqdm.write(f"❌ Failed after {max_retry} attempts. Data fetch for this day skipped.")
                break

        # Handle successful response (HTTP 200)
        elif response.status_code == 200:
            try:
                # Extract features from JSON
                results = response.json()
                results_dict = {
                    "date": results["date"],
                    "temp_min": results["temperature"]["min"],
                    "temp_max": results["temperature"]["max"],
                    "temp_avg": (results["temperature"]["min"] + results["temperature"]["max"]) / 2,
                    "cloud_cover": results["cloud_cover"]["afternoon"],
                    "humidity": results["humidity"]["afternoon"],
                    "precipitation": results["precipitation"]["total"],
                    "pressure": results["pressure"]["afternoon"],
                    "wind_speed": results["wind"]["max"]["speed"],
                    "wind_direction": results["wind"]["max"]["direction"]
                }

                # Save to cache
                with open(cache_file, "w") as f:
                    json.dump(results_dict, f)

                tqdm.write(f"ℹ️ data for day {day_str} cached")
                successful_call += 1
                break  # Exit retry loop on success

            except (KeyError, json.JSONDecodeError) as e:
                tqdm.write(f"❌ error when parsin day {day_str}: {e}")
                break

        # Handle other HTTP errors
        else:
            tqdm.write(f"❌ Error {response.status_code} on {day_str}")
            break

    # Rate limiting: wait 1 second between API calls
    time.sleep(1)

# --- Final check ---
if successful_call + cached_count == len(date_range):
    tqdm.write("✅ All days successful fetched!")
    tqdm.write(f"   Cached: {cached_count}, Fetched: {successful_call}")

Fetching weather data:   0%|          | 0/14 [00:00<?, ?it/s]

ℹ️ data for day 2023-05-08 cached
ℹ️ data for day 2023-05-09 cached
ℹ️ data for day 2023-05-10 cached
ℹ️ data for day 2023-05-11 cached
ℹ️ data for day 2023-05-12 cached
ℹ️ data for day 2023-05-13 cached
ℹ️ data for day 2023-05-14 cached
✅ All days successful fetched!
   Cached: 7, Fetched: 7


In [17]:
# =======================================
# MERGE JSON WEATHER DATA AND SAVE TO CSV
# =======================================
cache_path = Path(cache_dir)
json_files = cache_path.glob("weather_*.json")
data = []

for file in json_files:
    with open(file, "r") as f:
        data.append(json.load(f))

df = pd.DataFrame(data)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by= "date").reset_index(drop= True)

df.to_csv("../data/raw/weather.csv",index= False)


7